# Классификатор бот-трафика по поведению куки (Precision at Recall от 0.70)

Репозиторий и отчет содержат полное описание архитектуры, признакового пространства, экспериментальной работы и воспроизводимого пайплайна для решения задачи детекции автоматизированного сбора данных на платформе Авито.

---

## 1. Архитектура валидации и прогресс качества (Сравнение с Baseline)

Для полного исключения утечек данных во времени и достоверного моделирования боевого инференса применен строгий временной сплит Out-of-Time (OOT):
* Обучающая часть сплита: события с 6 по 13 апреля 2026 года (первая неделя, 6 773 куки, 545 ботов).
* Валидационная часть сплита: события с 14 по 19 апреля 2026 года (вторая неделя, 4 318 кук, 355 ботов). Длина валидационного отрезка (6 дней) полностью соответствует недельному объему скрытого теста (с 20 по 26 апреля, 4 909 кук).
* Неприменимость Random K-Fold: случайное разбиение по кукам приводит к заглядыванию в будущее (data leakage), так как модель выучивает разовые суточные волны сбора категорий и переобучается под временные всплески, показывая фиктивную точность выше 0.94, которая полностью деградирует на отложенном тесте.
* Adversarial Validation: классификатор различия train и test по извлеченным признакам показал ROC-AUC = 0.5005 (отсутствие временного дрейфа).
* Метрика: оценка производилась реализацией precision_at_recall из metric.py с обязательным учетом групповой склейки порогов при равенстве скоров.

### Сводная таблица прогресса метрики:
| Решение / Конфигурация | P@R от 0.70 (Val) | Прирост к Baseline | P@R от 0.70 (Test) |
| :--- | :---: | :---: | :---: |
| Уровень константы (доля ботов в выборке) | 0.0820 | - | 0.0939 |
| Quickstart Baseline (RandomForest, 2 фичи: n_events, item_nunique) | 0.0936 | 0.0000 | 0.1025 |
| Базовый ансамбль 3 моделей (LGBM + XGBoost + CatBoost) | 0.7500 | +0.6564 | - |
| Ансамбль с базовой защитой действий человека | 0.7880 | +0.6944 | - |
| Финальный ансамбль 30 моделей с пограничной калибровкой | 0.8032 | +0.7096 | 0.7902 |

Итоговая точность на валидации выросла в 8.5 раз по сравнению со стартовым бейзлайном. На скрытом тесте платформы достигнут результат 0.79024 (при уровне константы 0.09391, чистый абсолютный прирост +0.6963.). Расхождение между локальной валидацией (0.8032) и скрытым тестом составляет всего 1.3%, что доказывает отсутствие переобучения под валидационный сплит.

---

## 2. Признаковое пространство (Feature Engineering)

Все 90 признаков рассчитаны строго по событиям внутри суточного окна наблюдения: window_start_ts <= event_ts и event_ts < window_end_ts. События за пределами границ окна полностью исключены. Признаковое пространство сформировано из 7 функциональных блоков:

1. Сетевые повторы и retry-циклы:
   * raw_dup_counts (n_duplicates): число полных дубликатов событий, посчитанное строго внутри сессии каждой куки до дедупликации. Программные краулеры при нестабильном соединении и сбоях таймаута отправляют серии идентичных повторных запросов, что у живых людей практически отсутствует.
2. Динамика временных интервалов (dt):
   * Медиана, среднее, квантили (q25, q75), минимум и коэффициент вариации dt (узкая дисперсия пауз характерна для роботов).
   * Доля быстрых кликов с dt не более 1 секунды и нулевых задержек dt_zero_ratio.
   * Детекция жестких таймеров краулеров: is_sleep_1s (паузы от 0.95 до 1.05 сек) и is_sleep_2s (от 1.95 до 2.05 сек).
3. Поведение на карточках объявлений:
   * Время нахождения на карточке через отрицательный сдвиг до следующего действия: shift(-1) по таймстампу.
   * Логарифмическое среднее времени изучения карточки item_dwell_log_mean.
   * Плотность уникальных товаров к общему числу событий item_to_events_density.
4. Глубина каталога и поисковая пагинация:
   * Максимальная (search_page_max) и средняя страница выдачи (боты обходят листинги вглубь вплоть до 65 страницы, тогда как живые люди редко заходят дальше 3-4 страницы).
   * Доля глубоких страниц (страница 4 и выше) и монотонность шага пагинации step_page_ratio.
   * Отношение числа страниц поиска к уникальным запросам pages_per_query и доля пустых запросов empty_query_ratio (прямой обход рубрик без ввода текста).
5. Конверсии в целевые действия:
   * Конверсия карточек в показы телефонов и открытие чатов (contact_ratio).
   * Отношение свайпов фото (swipe_ratio) и добавлений в избранное (fav_ratio) к просмотрам карточек.
   * Комплексный объем действий человека human_actions_cnt (свайпы фото, избранное, логин, отправка сообщений).
   * Суточный ритм: ночная активность night_ratio (с 1 до 6 часов утра) и маркер непрерывного краулинга is_continuous_crawler (активность более 18 часов без перерывов свыше 3 часов).
6. Кинематика курсора (Pointer Kinematics):
   * Доля десктопных событий без координат мыши desktop_ptr_na_ratio (характерно для headless-эмуляторов и прямых сетевых запросов).
   * Среднее евклидово расстояние перемещения мыши ptr_dist_mean и число уникальных точек координат ptr_unique_pts.
   * Фиксация статичных кликов is_ptr_static (ptr_dist = 0) и синтетических вьюпортов малого разрешения is_small_viewport (ширина до 800 и высота до 600 пикселей).
7. Клиентское окружение и жизненный цикл:
   * Детекция HeadlessChrome, десктопного Linux без Android, версионность Chromium (chrome_ver_median) и маркер старых версий is_chrome_old (версия не выше 116).
   * Маркеры брендов мобильных устройств в строке User-Agent (Samsung, Xiaomi, Redmi, POCO, iPhone, Pixel, Huawei, Honor) и платформа is_pure_mobile.
   * Жизненный цикл: возраст куки в днях (cookie_age_days) от момента генерации cookie_created_at до начала окна наблюдения.

---

## 3. Моделирование, ансамблирование и устранение коллизий

Для подавления дисперсии единичных моделей построена трехуровневая композиция из 30 алгоритмов градиентного бустинга:
* 10 моделей LightGBM: n_estimators=800, learning_rate=0.025, num_leaves=31, max_depth=6, subsample=0.80, colsample_bytree=0.80, scale_pos_weight=1.25.
* 10 моделей XGBoost: n_estimators=750, learning_rate=0.025, max_depth=5, subsample=0.80, colsample_bytree=0.80, scale_pos_weight=1.60.
* 10 моделей CatBoost: iterations=750, learning_rate=0.035, depth=5, l2_leaf_reg=2.5.
* Устранение коллизий скоров: усреднение предсказаний по 10 независимым фиксированным сидам гарантирует получение 4 909 уникальных вещественных значений в тесте (sub.score.nunique() == len(test)). Это исключает потери метрики из-за групповой склейки порогов в функции precision_at_recall.
* Веса вероятностного блендинга: эмпирически доказанная пропорция 0.30 * LGBM + 0.50 * XGBoost + 0.20 * CatBoost (XGBoost с увеличенным весом редкого положительного класса обеспечивает наиболее контрастное разделение пограничных объектов).

---

## 4. Пограничная калибровка (Frontier Calibration)

Детальный анализ пограничной зоны рубежа полноты 70% (зона отсечки 249 ботов из 355 на валидации при пороге score = 0.161279) выявил четкое разделение классов:
* Пограничные боты: специализированные парсеры контактов с высокой конверсией (contact_ratio от 1.00 и выше).
* Пограничные люди (False Positives): активные пользователи смартфонов, глубоко листающие каталог (search_page_max от 6 до 10), но не собирающие телефоны массово (contact_ratio строго меньше 0.30, phone_cnt не более 2).

Применена точечная мультипликативная калибровка:
Score_final = Score_blend * exp(-alpha * Actions - beta * Favorites) * MobileDecay
где alpha = 0.070, beta = 0.100, а множитель MobileDecay = exp(-0.150) действует строго для пользователей с мобильными маркерами (brand == 1 или is_pure_mobile == 1) при contact_ratio строго меньше 0.30 и phone_cnt не более 2.
Калибровка показала нулевое задевание ботов на границе (0.0000) и сместила ложные срабатывания людей ниже порога отсечки, подняв точность с 0.7880 до 0.8032.

---

## 5. Каталог отрицательных экспериментов (Failed Experiments and Dead Ends)

В процессе оптимизации был проверен и аргументированно отклонен широкий спектр гипотез:

1. События показа капчи (captcha_shown):
   * Гипотеза: в первичном анализе у ботов частота капчи составляла 15.1% против 0.5% у людей.
   * Факт: детальный аудит показал, что все 7 928 событий капчи произошли строго после окончания суточного окна наблюдения (event_ts >= window_end_ts, средний временной лаг составил 19.3 часа).
   * Причина отказа: включение капчи является грубой утечкой данных из будущего. Признак полностью исключен.
2. Одиночная оптимизация гиперпараметров через Optuna:
   * Гипотеза: автоматический подбор параметров бустингов улучшит разделение.
   * Факт: Optuna урезала глубину LightGBM до 4 (вызвав сильное недообучение) и завысила random_strength в CatBoost до 2.0.
   * Причина отказа: подбор на одном сиде в отрыве от ансамбля нарушил баланс калибровки вероятностей, скор упал с 0.8032 до 0.7830. Откачено к сбалансированным параметрам.
3. Ослабление мобильного фильтра калибровки (contact_ratio меньше 0.60 без контроля phone_cnt):
   * Гипотеза: расширение фильтра защитит больше людей, глубоко листающих каталог.
   * Факт: скор упал с 0.8032 до 0.7830.
   * Причина отказа: фильтр задел реальных ботов-каталожников с умеренным сбором контактов. Боты опустились ниже порога, полнота упала ниже 70%, автопроверка была вынуждена снизить порог отсечки, впустив массив шума.
4. Ранговое усреднение (Percentile Rank Blending):
   * Гипотеза: перевод предсказаний моделей в процентили защитит от различий в калибровке шкал.
   * Факт: скор обвалился с 0.8032 до 0.3078 - 0.3324.
   * Причина отказа: процентили распределены строго равномерно от 0 до 1, сжимая естественный экспоненциальный зазор между ботами и людьми. Мультипликативная калибровка на равномерной шкале полностью разрушила разделение.
5. Степенное обострение вероятностей (score в степени gamma):
   * Факт: gamma = 0.85 дала 0.7757, gamma = 1.20 дала 0.7830, gamma = 1.00 дала оптимальные 0.7930 - 0.8032.
   * Причина отказа: исходная плотность распределения бленда оказалась математически оптимальной.
6. Яндекс.Браузер как жесткий маркер живого человека:
   * Гипотеза: YaBrowser используют исключительно реальные пользователи РФ-сегмента.
   * Факт: построчный аудит логов показал, что 3 из 11 пограничных ботов работали с заголовками YaBrowser/23.
   * Причина отказа: штраф за YaBrowser срезал скоры пограничным ботам, обвалив полноту.
7. Защита десктопных пользователей по физике мыши (d_decay):
   * Гипотеза: естественный разброс координат мыши доказывает присутствие живого человека.
   * Факт: скор упал до 0.7733.
   * Причина отказа: продвинутые скраперы на базе Chrome под macOS эмулировали системные движения курсора (ptr_dist_mean = 149 px). Штраф задел реальных ботов, нарушив баланс рубежа 70% полноты.
8. Глобальная дедупликация через ev.duplicated() по всей таблице:
   * Гипотеза: ускорение векторизации без использования groupby.apply().
   * Факт: валидационная метрика упала с 0.8032 до 0.7788.
   * Причина отказа: глобальный duplicated() помечал как дубликаты события разных пользователей, совпавшие по времени секунды и типу события. Признак n_duplicates замусорился у людей, лишив модель сильнейшего разделителя программных retry-циклов.
9. Синтаксический анализ поисковых запросов:
   * Факт: среднее число слов в запросе (1.98 у людей против 1.99 у ботов), доля цифр (19.1% против 18.7%) и доля латиницы (25.2% против 24.1%) совпали с точностью до десятых долей.
   * Причина отказа: современные краулеры генерируют естественные текстовые запросы из словарей листингов.
10. Время создания куки внутри окна (cookie_created_at >= window_start_ts):
    * Факт: 100% кук в обучающей и тестовой выборках были созданы строго до начала окна наблюдения. Признак выродился в константу.
11. Счетчики конкретных товарных категорий:
    * Факт: распределение по 10 рубрикам каталога оказалось искусственно сбалансировано организаторами (примерно по 12-13 тыс. событий на категорию с равной долей ботов 13-15%). Прямое кодирование категорий вело к переобучению.

---

## 6. Ограничения модели (Limitations)

1. Эволюция сигнатур автоматизации: при переходе краулеров на пулы заголовков актуальных мобильных приложений Авито с нулевым сбором телефонов потребуется перекалибровка порогов эвристического дисконта.
2. Аппаратное отсутствие координат курсора на мобильных устройствах: на смартфонах события pointer_x и pointer_y не генерируются аппаратно, поэтому разделение мобильного трафика опирается преимущественно на динамику таймингов и граф переходов по листингу.
3. Фиксация суточного окна наблюдения: агрегаты интенсивности и квантили пауз калиброваны строго под 24-часовое окно; при переходе на скользящие окна малой длины (1-2 часа) потребуется нормализация признаков на фактическую продолжительность сессии.

---

## 7. Воспроизводимость и окружение

* Язык и среда исполнения: Python 3.12.4 в Jupyter Notebook
* Операционная система: Windows 10 (версия Windows-10-10.0.19045-SP0)
* Процессор: Intel
* Режим выполнения: CPU only (расчет через n_jobs=-1, thread_count=-1). Полное время выполнения блокнота: 3-4 минуты.
* Фиксация случайности: зафиксированы seed_const = 42 (random.seed, np.random.seed, PYTHONHASHSEED="42") и набор из 10 детерминированных random_state для каждого бустинга: [42, 100, 2024, 777, 999, 1337, 314, 2718, 555, 888].
* Точные версии библиотек текущей сессии (зафиксированы в requirements.txt через ==):
  * numpy==1.26.4
  * pandas==3.0.3
  * scikit-learn==1.8.0
  * lightgbm==4.6.0
  * xgboost==3.4.1
  * catboost==1.2.10
  * scipy==1.17.1

### Контрольные проверки сформированного файла (Pre-flight Audit):
Скрипт блокнота перед сохранением submission.csv выполняет автоматические проверки:
* Число строк строго равно 4 909 (полное соответствие test.csv).
* Полное отсутствие пропущенных значений (NaN).
* Все скоры строго лежат в диапазоне от 0 до 1.
* Число уникальных значений равно 4 909 (100% уникальность скоров без коллизий, гарантирующая непрерывное ранжирование).

In [ ]:
# вынос всех библиотек в начало скрипта для чистоты окружения
import os
import re
import random
import warnings
import numpy as np
import pandas as pd
from metric import precision_at_recall
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

# подавление некритичных сервисных предупреждений библиотек
warnings.filterwarnings("ignore")

seed_const = 42
os.environ["PYTHONHASHSEED"] = str(seed_const)
random.seed(seed_const)
np.random.seed(seed_const)

df_tr = pd.read_csv('data/train.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
df_te = pd.read_csv('data/test.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
df_ev = pd.read_csv('data/events.csv.gz', parse_dates=['event_ts'])

print("размеры исходных таблиц:", df_tr.shape, df_te.shape, df_ev.shape)
print("доля ботов в train:", df_tr.target.mean().round(4))

def filter_win_ev(ev_df: pd.DataFrame, meta_df: pd.DataFrame) -> pd.DataFrame:
    m = ev_df.merge(meta_df[['cookie_id', 'window_start_ts', 'window_end_ts']], on='cookie_id')
    return m[(m.event_ts >= m.window_start_ts) & (m.event_ts < m.window_end_ts)]

win_tr = filter_win_ev(df_ev, df_tr)
win_te = filter_win_ev(df_ev, df_te)
print("число событий внутри окна: train =", len(win_tr), "| test =", len(win_te))

d:\Anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
d:\Anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


размеры исходных таблиц: (11091, 5) (4909, 4) (328905, 14)
доля ботов в train: 0.0811
число событий внутри окна: train = 198436 | test = 89690


In [ ]:
def build_features(ev_in_win: pd.DataFrame, meta: pd.DataFrame) -> pd.DataFrame:
    ev = ev_in_win.copy()
    
    # подсчет полных дубликатов строк внутри конкретной куки до их удаления
    dup_cnt = ev.groupby('cookie_id').apply(lambda d: d.duplicated().sum())
    ev.drop_duplicates(inplace=True)
    ev.sort_values(['cookie_id', 'event_ts'], inplace=True)

    # нормализация типов платформ к единому нижнему регистру
    ev['plat_low'] = ev['platform'].astype(str).str.lower()
    ev['is_desk'] = ev['plat_low'].isin(['desktop', 'web']).astype(np.int8)
    ev['is_mob'] = ev['plat_low'].isin(['android', 'ios', 'iphone']).astype(np.int8)

    # поиск сигнатур ботнетов headless браузеров и специфических операционных систем
    ua = ev['user_agent'].fillna('').astype(str)
    ev['ua_is_headless'] = ua.str.contains('headless|phantomjs', case=False, regex=True).astype(np.int8)
    ev['ua_is_linux'] = (ua.str.contains('Linux', case=False, regex=True) & ~ua.str.contains('Android', case=False, regex=True)).astype(np.int8)
    ev['ua_is_yabrowser'] = ua.str.contains('YaBrowser', case=False, regex=True).astype(np.int8)
    ev['ua_is_firefox'] = ua.str.contains('Firefox', case=False, regex=True).astype(np.int8)
    ev['ua_is_mac'] = ua.str.contains('Macintosh|Mac OS', case=False, regex=True).astype(np.int8)
    ev['ua_is_windows'] = ua.str.contains('Windows NT', case=False, regex=True).astype(np.int8)
    ev['ua_has_phone_brand'] = ua.str.contains(r'SM-|Redmi|Xiaomi|POCO|iPhone|Pixel|Huawei|HONOR', case=False, regex=True).astype(np.int8)

    # извлечение версии chromium
    def get_chrome_ver(s: str) -> float:
        m = re.search(r'Chrome/(\d+)\.', s)
        return float(m.group(1)) if m else np.nan

    ev['chrome_ver'] = ua.apply(get_chrome_ver)

    # разница во времени между текущим и предыдущим событием для оценки темпа сессии
    ev['prev_ts'] = ev.groupby('cookie_id')['event_ts'].shift(1)
    ev['dt'] = (ev['event_ts'] - ev['prev_ts']).dt.total_seconds()
    
    # расчет длительности изучения карточки объявления через отрицательный сдвиг
    ev['next_ts'] = ev.groupby('cookie_id')['event_ts'].shift(-1)
    ev['dwell_sec'] = (ev['next_ts'] - ev['event_ts']).dt.total_seconds()
    ev['is_item_view'] = (ev['event_name'] == 'item_view').astype(np.int8)
    ev['item_dwell'] = np.where(ev['is_item_view'] == 1, ev['dwell_sec'], np.nan)
    ev['item_dwell_log'] = np.log1p(np.maximum(0.0, ev['item_dwell']))

    # детекция искусственных фиксированных задержек sleep в скриптах сбора с допуском 50мс
    ev['is_sleep_1s'] = ((ev['dt'] >= 0.95) & (ev['dt'] <= 1.05)).astype(np.int8)
    ev['is_sleep_2s'] = ((ev['dt'] >= 1.95) & (ev['dt'] <= 2.05)).astype(np.int8)

    # евклидово расстояние смещения курсора мыши между соседними событиями
    ev['prev_px'] = ev.groupby('cookie_id')['pointer_x'].shift(1)
    ev['prev_py'] = ev.groupby('cookie_id')['pointer_y'].shift(1)
    ev['ptr_dist'] = np.sqrt((ev['pointer_x'] - ev['prev_px'])**2 + (ev['pointer_y'] - ev['prev_py'])**2)
    ev['is_ptr_static'] = ((ev['ptr_dist'] == 0.0) & ev['pointer_x'].notna()).astype(np.int8)
    ev['desktop_ptr_is_na'] = (ev['is_desk'] == 1) & ev['pointer_x'].isna()

    ev['hour'] = ev['event_ts'].dt.hour
    ev['prev_page'] = ev.groupby('cookie_id')['search_page'].shift(1)
    # проверка последовательного перелистывания страниц пагинации шаг за шагом
    ev['is_step_page'] = ((ev['search_page'] - ev['prev_page']) == 1.0).astype(np.int8)

    ev['query_str'] = ev['search_query'].fillna('').astype(str)
    ev['query_len'] = ev['query_str'].str.len()

    grp = ev.groupby('cookie_id')
    feats = pd.DataFrame(index=meta['cookie_id'])

    # базовые объемы действий и плотность запросов в единицу времени
    feats['n_events'] = grp.size()
    feats['n_duplicates'] = dup_cnt.reindex(feats.index, fill_value=0)
    feats['duration_sec'] = (grp['event_ts'].max() - grp['event_ts'].min()).dt.total_seconds()
    feats['events_per_min'] = feats['n_events'] / ((feats['duration_sec'] + 60.0) / 60.0)
    feats['item_dwell_log_mean'] = grp['item_dwell_log'].mean().fillna(-1.0)

    # паттерны распределения активности по времени суток
    feats['active_hours'] = grp['hour'].nunique()
    feats['night_ratio'] = grp['hour'].apply(lambda s: ((s >= 1) & (s <= 6)).mean())
    feats['hour_std'] = grp['hour'].std().fillna(0.0)
    feats['max_gap_hours'] = grp['dt'].max().fillna(0.0) / 3600.0
    feats['span_hours'] = feats['duration_sec'] / 3600.0
    # признак непрерывного парсинга без сна более 18 часов с малыми перерывами
    feats['is_continuous_crawler'] = (
        (feats['span_hours'] >= 18.0) & 
        (feats['max_gap_hours'] <= 3.0)
    ).astype(np.int8)

    # квантили пауз между запросами показывают механическую равномерность бота против человека
    feats['dt_median'] = grp['dt'].median()
    feats['dt_mean'] = grp['dt'].mean()
    feats['dt_std'] = grp['dt'].std().fillna(0.0)
    feats['dt_min'] = grp['dt'].min()
    feats['dt_q25'] = grp['dt'].quantile(0.25)
    feats['dt_q75'] = grp['dt'].quantile(0.75)
    feats['dt_zero_ratio'] = grp['dt'].apply(lambda s: (s == 0.0).mean())
    feats['dt_sub_1s_ratio'] = grp['dt'].apply(lambda s: (s <= 1.0).mean())
    feats['dt_coef_var'] = feats['dt_std'] / (feats['dt_mean'] + 1e-4)
    feats['sleep_timer_ratio'] = grp['is_sleep_1s'].mean() + grp['is_sleep_2s'].mean()

    # распределение частот конкретных типов действий в сессии
    ev_cnt = grp['event_name'].value_counts().unstack(fill_value=0).reindex(feats.index, fill_value=0)
    ev_types = [
        'item_view', 'search_results_view', 'photo_swipe', 'favorite_add',
        'seller_page_view', 'contact_phone_show', 'captcha_shown',
        'login', 'contact_chat_open', 'contact_message_sent'
    ]
    for e in ev_types:
        cnt = ev_cnt[e] if e in ev_cnt.columns else 0
        feats[f'cnt_{e}'] = cnt
        feats[f'ratio_{e}'] = cnt / feats['n_events']

    # комплексный объем действий свойственных реальному человеку
    feats['human_actions_cnt'] = (
        feats['cnt_photo_swipe'] + 
        feats['cnt_favorite_add'] + 
        feats['cnt_login'] + 
        feats['cnt_contact_message_sent']
    )
    feats['human_actions_ratio'] = feats['human_actions_cnt'] / feats['n_events']

    # конверсии карточек в целевые действия
    safe_it = feats['cnt_item_view'] + 1e-4
    feats['contact_ratio'] = (feats['cnt_contact_phone_show'] + feats['cnt_contact_chat_open']) / safe_it
    feats['swipe_ratio'] = feats['cnt_photo_swipe'] / safe_it
    feats['fav_ratio'] = feats['cnt_favorite_add'] / safe_it

    # характеристики взаимодействия с поисковой выдачей
    feats['search_page_mean'] = grp['search_page'].mean().fillna(0.0)
    feats['search_page_max'] = grp['search_page'].max().fillna(0.0)
    feats['search_page_std'] = grp['search_page'].std().fillna(0.0)
    feats['step_page_ratio'] = grp['is_step_page'].mean()
    feats['page_ge_4_ratio'] = grp['search_page'].apply(lambda s: (s >= 4.0).mean())
    feats['query_len_mean'] = grp['query_len'].mean()
    feats['empty_query_ratio'] = grp['search_query'].apply(lambda s: s.isna().mean())
    feats['query_nunique'] = grp['search_query'].nunique()
    feats['pages_per_query'] = feats['cnt_search_results_view'] / (feats['query_nunique'] + 1e-4)

    # пространственная кинематика мыши
    feats['ptr_not_na_ratio'] = grp['pointer_x'].apply(lambda s: s.notna().mean())
    feats['ptr_dist_mean'] = grp['ptr_dist'].mean().fillna(0.0)
    feats['ptr_static_ratio'] = grp['is_ptr_static'].mean()
    feats['ptr_unique_pts'] = grp.apply(lambda d: len(set(zip(d['pointer_x'].dropna(), d['pointer_y'].dropna()))))
    
    desk_cnt = grp['is_desk'].sum()
    feats['desktop_ptr_na_ratio'] = (grp['desktop_ptr_is_na'].sum() / (desk_cnt + 1e-4)).fillna(0.0)
    feats['ptr_x_max'] = grp['pointer_x'].max().fillna(0.0)
    feats['ptr_y_max'] = grp['pointer_y'].max().fillna(0.0)
    # проверка на запуск в тестовых окнах фиксированного малого разрешения
    feats['is_small_viewport'] = (
        (feats['ptr_x_max'] > 0.0) & 
        (feats['ptr_x_max'] <= 800.0) & 
        (feats['ptr_y_max'] <= 600.0)
    ).astype(np.int8)

    # разнообразие объектов и категорий внутри окна
    feats['item_nunique'] = grp['item_id'].nunique()
    feats['category_nunique'] = grp['item_category'].nunique()
    feats['location_nunique'] = grp['item_location'].nunique()
    feats['item_breadth_ratio'] = feats['item_nunique'] / safe_it
    feats['top_category_ratio'] = grp['item_category'].apply(
        lambda s: s.value_counts(normalize=True).iloc[0] if len(s.dropna()) > 0 else 0.0
    )
    feats['item_to_events_density'] = feats['item_nunique'] / feats['n_events']

    # доли просмотров в ключевых целевых категориях массового сбора данных
    cat_cnt = grp['item_category'].value_counts().unstack(fill_value=0).reindex(feats.index, fill_value=0)
    for cat in ['kvartiry_prodazha', 'telefony', 'elektronika', 'odezhda']:
        cnt = cat_cnt[cat] if cat in cat_cnt.columns else 0
        feats[f'cat_ratio_{cat}'] = cnt / safe_it

    # агрегация клиентского окружения
    feats['chrome_ver_median'] = grp['chrome_ver'].median().fillna(120.0)
    feats['is_chrome_old'] = (feats['chrome_ver_median'] <= 116.0).astype(np.int8)
    feats['has_headless'] = grp['ua_is_headless'].max()
    feats['has_linux'] = grp['ua_is_linux'].max()
    feats['has_mac'] = grp['ua_is_mac'].max()
    feats['has_windows'] = grp['ua_is_windows'].max()
    feats['has_yabrowser'] = grp['ua_is_yabrowser'].max()
    feats['has_firefox'] = grp['ua_is_firefox'].max()
    feats['ua_nunique'] = grp['user_agent'].nunique()
    feats['has_phone_brand'] = grp['ua_has_phone_brand'].max()

    plat_cnt = grp['plat_low'].value_counts().unstack(fill_value=0).reindex(feats.index, fill_value=0)
    for p in ['desktop', 'web', 'android', 'ios', 'iphone']:
        cnt = plat_cnt[p] if p in plat_cnt.columns else 0
        feats[f'plat_ratio_{p}'] = cnt / feats['n_events']
    feats['is_pure_mobile'] = (grp['is_mob'].min() == 1).astype(np.int8)

    # вычисление возраста куки до момента старта окна наблюдения
    feats = feats.reset_index().merge(
        meta[['cookie_id', 'cookie_created_at', 'window_start_ts']], 
        on='cookie_id', 
        how='left'
    )
    feats['cookie_age_days'] = (feats['window_start_ts'] - feats['cookie_created_at']).dt.total_seconds() / 86400.0
    feats.drop(columns=['cookie_created_at', 'window_start_ts'], inplace=True)

    return feats.fillna(0.0)

print("генерация признакового пространства...")
x_tr = build_features(win_tr, df_tr)
x_te = build_features(win_te, df_te)
y_tr = df_tr.target.values

f_cols = [c for c in x_tr.columns if c != 'cookie_id']
print(f"извлечено признаков: {len(f_cols)}")

генерация признакового пространства...
извлечено признаков: 90


In [ ]:
# out of time сплит
val_mask = df_tr.window_start_ts.ge('2026-04-14').values
tr_mask = ~val_mask
base_cols = ['n_events', 'item_nunique']

# обучение стартовой базовой модели случайного леса из quickstart
rf_base = RandomForestClassifier(n_estimators=300, min_samples_leaf=3, random_state=0)
rf_base.fit(x_tr.loc[tr_mask, base_cols], y_tr[tr_mask])
val_p_base = rf_base.predict_proba(x_tr.loc[val_mask, base_cols])[:, 1]
score_base = precision_at_recall(y_tr[val_mask], val_p_base)

print(f"P@R0.7 на валидации с 14 апреля (стартовый baseline): {score_base:.4f}")
print(f"доля ботов на валидации (уровень константы):        {y_tr[val_mask].mean():.4f}")

P@R0.7 на валидации с 14 апреля (стартовый baseline): 0.0936
доля ботов на валидации (уровень константы):        0.0822


In [ ]:
seeds = [42, 100, 2024, 777, 999, 1337, 314, 2718, 555, 888]

# оптимальные веса вероятностей моделей
w_lgb, w_xgb, w_cat = 0.30, 0.50, 0.20
# коэффициенты экспоненциального затухания за типично человеческие действия
a_dec = 0.070
b_dec = 0.100
m_dec = 0.150

# 1. замер качества ансамбля на отложенном временном сплите
xv_tr, yv_tr = x_tr.loc[tr_mask, f_cols], y_tr[tr_mask]
xv_val, yv_val = x_tr.loc[val_mask, f_cols], y_tr[val_mask]

val_p_lgb = np.zeros(len(xv_val))
val_p_xgb = np.zeros(len(xv_val))
val_p_cat = np.zeros(len(xv_val))

print("обучение 10 сидов на сплите валидации...")
for s in seeds:
    lgb_v = LGBMClassifier(
        n_estimators=800, learning_rate=0.025, num_leaves=31, max_depth=6,
        subsample=0.80, colsample_bytree=0.80, scale_pos_weight=1.25,
        random_state=s, n_jobs=-1, verbose=-1
    )
    lgb_v.fit(xv_tr, yv_tr)
    val_p_lgb += lgb_v.predict_proba(xv_val)[:, 1] / len(seeds)

    xgb_v = XGBClassifier(
        n_estimators=750, learning_rate=0.025, max_depth=5,
        subsample=0.80, colsample_bytree=0.80, scale_pos_weight=1.60,
        random_state=s, n_jobs=-1, eval_metric='logloss'
    )
    xgb_v.fit(xv_tr, yv_tr)
    val_p_xgb += xgb_v.predict_proba(xv_val)[:, 1] / len(seeds)

    cat_v = CatBoostClassifier(
        iterations=750, learning_rate=0.035, depth=5,
        l2_leaf_reg=2.5, random_seed=s, thread_count=-1, verbose=0
    )
    cat_v.fit(xv_tr, yv_tr)
    val_p_cat += cat_v.predict_proba(xv_val)[:, 1] / len(seeds)

# взвешенное линейное объединение вероятностей трех семейств бустинга
val_blend = w_lgb * val_p_lgb + w_xgb * val_p_xgb + w_cat * val_p_cat

v_act = xv_val['human_actions_cnt'].values
v_fav = xv_val['cnt_favorite_add'].values
v_brand = xv_val['has_phone_brand'].values
v_pmob = xv_val['is_pure_mobile'].values
v_crat = xv_val['contact_ratio'].values
v_pcnt = xv_val['cnt_contact_phone_show'].values

# дисконтирование скора за действия которые боты практически никогда не делают
v_h_dec = np.exp(-a_dec * v_act - b_dec * v_fav)
# безопасное выделение реальных мобильных покупателей без массового сбора контактов
v_is_mob = ((v_brand == 1) | (v_pmob == 1)) & (v_crat < 0.30) & (v_pcnt <= 2)
v_m_dec = np.where(v_is_mob, np.exp(-m_dec), 1.0)

# применение калибровки для точечного смещения ложноположительных людей ниже порога
val_final_p = val_blend * v_h_dec * v_m_dec
val_score = precision_at_recall(yv_val, val_final_p)

print(f"\n=======================================================")
print(f"ИТОГОВАЯ МЕТРИКА P@R>=0.70 НА ВАЛИДАЦИИ (10 СИДОВ): {val_score:.4f}")
print(f"СТАРТОВЫЙ BASELINE (НА ЭТОМ ЖЕ СПЛИТЕ):             {score_base:.4f}")
print(f"ЧИСТЫЙ АБСОЛЮТНЫЙ ПРИРОСТ:                         +{val_score - score_base:.4f}")
print(f"=======================================================\n")

# 2. обучение 30 финальных моделей на 100% размеченных данных train
print("=== ОБУЧЕНИЕ 30 МОДЕЛЕЙ НА 100% ДАННЫХ И СБОРКА SUBMISSION.CSV ===")
te_p_lgb = np.zeros(len(df_te))
te_p_xgb = np.zeros(len(df_te))
te_p_cat = np.zeros(len(df_te))

for s in seeds:
    lgb = LGBMClassifier(
        n_estimators=800, learning_rate=0.025, num_leaves=31, max_depth=6,
        subsample=0.80, colsample_bytree=0.80, scale_pos_weight=1.25,
        random_state=s, n_jobs=-1, verbose=-1
    )
    lgb.fit(x_tr[f_cols], y_tr)
    te_p_lgb += lgb.predict_proba(x_te[f_cols])[:, 1] / len(seeds)

    xgb = XGBClassifier(
        n_estimators=750, learning_rate=0.025, max_depth=5,
        subsample=0.80, colsample_bytree=0.80, scale_pos_weight=1.60,
        random_state=s, n_jobs=-1, eval_metric='logloss'
    )
    xgb.fit(x_tr[f_cols], y_tr)
    te_p_xgb += xgb.predict_proba(x_te[f_cols])[:, 1] / len(seeds)

    cat = CatBoostClassifier(
        iterations=750, learning_rate=0.035, depth=5,
        l2_leaf_reg=2.5, random_seed=s, thread_count=-1, verbose=0
    )
    cat.fit(x_tr[f_cols], y_tr)
    te_p_cat += cat.predict_proba(x_te[f_cols])[:, 1] / len(seeds)

te_blend = w_lgb * te_p_lgb + w_xgb * te_p_xgb + w_cat * te_p_cat

te_act = x_te['human_actions_cnt'].values
te_fav = x_te['cnt_favorite_add'].values
te_brand = x_te['has_phone_brand'].values
te_pmob = x_te['is_pure_mobile'].values
te_crat = x_te['contact_ratio'].values
te_pcnt = x_te['cnt_contact_phone_show'].values

te_h_dec = np.exp(-a_dec * te_act - b_dec * te_fav)
te_is_mob = ((te_brand == 1) | (te_pmob == 1)) & (te_crat < 0.30) & (te_pcnt <= 2)
te_m_dec = np.where(te_is_mob, np.exp(-m_dec), 1.0)

te_final_p = te_blend * te_h_dec * te_m_dec

# сборка итогового датафрейма для сабмита
sub = pd.DataFrame({
    'cookie_id': x_te.cookie_id,
    'score': te_final_p,
})

# структура файла перед отправкой
assert len(sub) == len(df_te), "число строк не совпадает с test.csv"
assert sub.score.between(0, 1).all(), "найдены скоры выходящие за границы от 0 до 1"
assert not sub.score.isna().any(), "обнаружены пропущенные значения nan"
assert sub.score.nunique() == len(df_te), "обнаружены одинаковые скоры приводящие к склейке порогов"

# запись финального файла
sub.to_csv('submission.csv', index=False)

print("\n=== ФАЙЛ SUBMISSION.CSV УСПЕШНО СФОРМИРОВАН И СОХРАНЕН ===")
print("число строк:", len(sub))
print("уникальных скоров:", sub.score.nunique())
print("минимальный скор:", round(sub.score.min(), 6), "| максимальный:", round(sub.score.max(), 6))
print("\nпервые 5 строк submission.csv:")
print(sub.head())

обучение 10 сидов на сплите валидации...

ИТОГОВАЯ МЕТРИКА P@R>=0.70 НА ВАЛИДАЦИИ (10 СИДОВ): 0.8032
СТАРТОВЫЙ BASELINE (НА ЭТОМ ЖЕ СПЛИТЕ):             0.0936
ЧИСТЫЙ АБСОЛЮТНЫЙ ПРИРОСТ:                         +0.7096

=== ОБУЧЕНИЕ 30 МОДЕЛЕЙ НА 100% ДАННЫХ И СБОРКА SUBMISSION.CSV ===

=== ФАЙЛ SUBMISSION.CSV УСПЕШНО СФОРМИРОВАН И СОХРАНЕН ===
число строк: 4909
уникальных скоров: 4909
минимальный скор: 1.5e-05 | максимальный: 0.998724

первые 5 строк submission.csv:
             cookie_id     score
0  ck_315fb710a0e371e7  0.000080
1  ck_a76ee3b3e3e522fd  0.132492
2  ck_94c9a4d382689e82  0.082422
3  ck_8eaf9509ad9462a0  0.000495
4  ck_9a88a5a989cb5bc6  0.005879
